To make our RAG system work, we need a way to translate our text chunks into math. This is what an Embedding Model does. It takes a piece of text and converts it into a long list of numbers (a vector) -> similarity is then how similar their vectors are

The magic here is that these numbers represent the semantic meaning of the text. If two chunks of text have similar meanings (like "puppy" and "dog"), their lists of numbers will be mathematically close to each other. If they are completely different ("puppy" and "skyscraper"), their numbers will be far apart.

By converting our document chunks and our search queries into these number vectors, we can use basic math to find the chunks that best answer the question.

The difference between different embedding models: <br>
**Different embedding models are built for different jobs, and picking the right one is a balancing act between speed, accuracy, and hardware limits. Here is a breakdown of what different models are good for:** <br>
* Lightweight & Fast (e.g., all-MiniLM-L6-v2) -> lower 'resolution'
* Heavyweight & High Accuracy (e.g., nomic-embed-text, BGE-Large) -> higher precision, good for complex documents
* Domain-Specific Models -> Multilingual, Code/Math (jina-embeddings-v2-base-code)

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Initialize the HuggingFace embedding model
# This might take a few seconds the first time as it downloads the model weights
hf_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Let's create a test sentence
test_text = "Local RAG systems protect my privacy."

# 3. Embed the text (translate it into math)
vector_result = hf_embeddings.embed_query(test_text)

# 4. Look at the results!
print(f"The text was converted into a list of {len(vector_result)} numbers.")
print("Here are the first 5 numbers in the vector:")
print(vector_result[:5])

/Users/pierson.chu/Downloads/github/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/pierson.chu/Downloads/github/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14361.67it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The text was converted into a list of 384 numbers.
Here are the first 5 numbers in the vector:
[-0.02964956685900688, 0.08204346150159836, 0.059863731265068054, -0.0337044782936573, 0.012122214771807194]


In [2]:
from langchain_community.embeddings import OllamaEmbeddings

# 1. Initialize the Ollama embedding model
# This connects to the Ollama service running on your machine
ollama_embeddings = OllamaEmbeddings(model="nomic-embed-text")

# 2. We'll use the exact same test sentence from before
test_text = "Local RAG systems protect my privacy."

# 3. Embed the text using Ollama
ollama_vector_result = ollama_embeddings.embed_query(test_text)

# 4. Look at the results!
print(f"Ollama converted the text into a list of {len(ollama_vector_result)} numbers.")
print("Here are the first 5 numbers from Ollama's vector:")
print(ollama_vector_result[:5])

/var/folders/gp/9snl51jj70dd01jy_yq1bcxr0000gp/T/ipykernel_29614/4105335826.py:5: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  ollama_embeddings = OllamaEmbeddings(model="nomic-embed-text")


Ollama converted the text into a list of 768 numbers.
Here are the first 5 numbers from Ollama's vector:
[0.08483155816793442, 1.3241008520126343, -3.3641467094421387, 0.33879899978637695, 1.2255029678344727]


Benchmarking

When you build a RAG system, you will eventually need to embed hundreds or thousands of document chunks. If a model takes 1 second per chunk, 1,000 chunks will take over 15 minutes! If it takes 0.1 seconds, you are done in less than 2 minutes.

We are going to use Jupyter Notebook's built-in %%time magic command. It acts like a stopwatch for the entire cell.

Important: For the %%time command to work, it must be the very first line in your Jupyter cell.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
# 1. Create a dummy text file to practice on
sample_text = """
Retrieval-Augmented Generation (RAG) is a technique that bridges the gap between 
a large language model's general knowledge and your specific, private data. 
By using local models like Llama 3.1 and local embeddings, you ensure that 
sensitive information never leaves your machine. The first step in any RAG 
pipeline is data ingestion, which involves loading documents and splitting 
them into chunks. Proper chunking is critical because if chunks are too small, 
they lose semantic meaning. If they are too large, they might exceed the 
model's context window or dilute the specific answer you are looking for.
""" * 10 # Multiplying by 10 to make the text long enough to chunk!

with open("local_rag_guide.txt", "w") as file:
    file.write(sample_text)

# 2. Load the document using LangChain
    # Building a RAG system from scratch requires a lot of repetitive, messy code: opening files, writing loops to break up text, 
    # formatting prompts, sending data to the LLM, and parsing the output. 
    # LangChain provides standardized, pre-built functions for almost everything you need to do with an LLM. 
    # Instead of writing custom code to handle a PDF, a Word Doc, or a web page, LangChain gives you uniform tools to handle them all the exact same way.
from langchain_community.document_loaders import TextLoader

# If you just use standard Python to open a text file, you just get a giant string of text. 
# But in a RAG system, we need to keep track of where that text came from (so we can cite our sources later!).
    # The TextLoader reads your local .txt file and packages it into a standard LangChain Document object organising your data into two parts:
        # page_content: the actual text content of the document
        # metadata: a dictionary where you can store any extra information about the document (like the file path, the title, or the date it was created)
loader = TextLoader("local_rag_guide.txt")
# The .load() method is the trigger.
docs = loader.load()

# 3. Verify it loaded correctly
print(f"Successfully loaded {len(docs)} document(s).")
print(f"Total character count: {len(docs[0].page_content)}")

# 1. Set up three different chunking strategies
small_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
medium_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
large_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

# 2. Apply them to our original loaded documents
small_chunks = small_splitter.split_documents(docs)
medium_chunks = medium_splitter.split_documents(docs)
large_chunks = large_splitter.split_documents(docs)


Successfully loaded 1 document(s).
Total character count: 6160


In [ ]:
%%time
# Grab up to 50 of the medium chunks we made in Lesson 1
# (If your document was shorter, it will just grab what it has)
test_chunks = medium_chunks[:50]
texts_to_embed = [chunk.pagentent for chunk in test_chunks]

print(f"Embedding {len(texts_to_embed)} chunks with HuggingFace (MiniLM)...")
# In LangChain, embedding models generally have two distinct commands that we use for two different stages of the RAG pipeline:
    # embed_query() -> For the User's Question -> Input: A single string of text (e.g., "What is the refund policy?").
    # embed_documents() -> For the Chunks of Text -> Input: A list of strings (e.g., ["Our refund policy is...", "Refunds are processed within..."])
hf_results = hf_embeddings.embed_documents(texts_to_embed)
print("Done!")

Embedding 20 chunks with HuggingFace (MiniLM)...
Done!
CPU times: user 91.1 ms, sys: 106 ms, total: 198 ms
Wall time: 1.53 s


In [6]:
%%time
print(f"Embedding {len(texts_to_embed)} chunks with Ollama (Nomic)...")
ollama_results = ollama_embeddings.embed_documents(texts_to_embed)
print("Done!")

Embedding 20 chunks with Ollama (Nomic)...
Done!
CPU times: user 31 ms, sys: 10.6 ms, total: 41.5 ms
Wall time: 996 ms
